In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from google.colab import files, output
from IPython.display import display, HTML
import io
import base64
from PIL import Image

print("resmini yükle...")
uploaded = files.upload()
file_name = next(iter(uploaded))
image_data = uploaded[file_name]
image = np.array(Image.open(io.BytesIO(image_data)))

if len(image.shape) == 3 and image.shape[2] == 4:
    image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
elif len(image.shape) == 3 and image.shape[2] == 3:
    pass # Zaten RGB

_, buffer = cv2.imencode('.jpg', cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
img_str = base64.b64encode(buffer).decode('utf-8')

kuyucuklar = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 
              'E1', 'E2', 'F1', 'F2', 'G1', 'G2', 'H1', 'H2']

html_code = f'''
<style>
    #canvas-container {{ position: relative; display: inline-block; cursor: crosshair; }}
    canvas {{ border: 2px solid #ccc; max-width: 100%; height: auto; }}
    #info-box {{ font-size: 18px; font-weight: bold; background-color: #f1f8ff; padding: 15px; border: 1px solid #c8e1ff; margin-bottom: 10px; border-radius: 5px; }}
    .highlight {{ color: #d73a49; font-size: 22px; }}
    #controls {{ margin-top: 10px; margin-bottom: 15px; background: #fff3cd; padding: 10px; border-radius: 5px; border: 1px solid #ffeeba; }}
    input[type=range] {{ width: 300px; vertical-align: middle; }}
</style>

<div id="info-box">
    Adım 1: Lütfen fare ile <span id="current-well" class="highlight">A1</span> kuyucuğunun tam ortasına tıklayın. 
    (<span id="count">0</span> / 16)
</div>

<div id="controls">
    <label for="radiusSlider" style="font-size: 16px; font-weight: bold;">(Yarıçap): </label>
    <input type="range" id="radiusSlider" min="5" max="100" value="25">
    <span id="radiusValue" style="font-size: 18px; font-weight: bold; color: #856404;">25 px</span>
    <p style="margin: 5px 0 0 0; font-size: 14px; font-weight: normal;">* Sadece sıvının olduğu renkli alanı seçecek şekilde boyutu ayarlayın. Daireyi yerleştirdikten sonra boyutu tekrar değiştirebilirsiniz.</p>
</div>

<div id="canvas-container">
    <canvas id="imgCanvas"></canvas>
</div>

<script>
    var img = new Image();
    img.src = "data:image/jpeg;base64,{img_str}";
    var canvas = document.getElementById('imgCanvas');
    var ctx = canvas.getContext('2d');
    var coords = []; // Her tıklama için {{x, y, r}} saklayacak
    var wells = {kuyucuklar};
    
    var radiusSlider = document.getElementById('radiusSlider');
    var radiusValue = document.getElementById('radiusValue');
    var mouseX = 0, mouseY = 0;
    var isMouseOver = false;

    // Python'a veriyi göndermek için Colab promise yapısı
    var resolvePromise;
    var promise = new Promise(function(resolve) {{ resolvePromise = resolve; }});

    // Slider değiştiğinde önizleme dairesini güncelle
    radiusSlider.oninput = function() {{
        radiusValue.innerText = this.value + " px";
        drawAll();
    }}

    img.onload = function() {{
        canvas.width = img.width;
        canvas.height = img.height;
        drawAll();
    }};

    canvas.onmousemove = function(e) {{
        var rect = canvas.getBoundingClientRect();
        var scaleX = canvas.width / rect.width;
        var scaleY = canvas.height / rect.height;
        mouseX = Math.round((e.clientX - rect.left) * scaleX);
        mouseY = Math.round((e.clientY - rect.top) * scaleY);
        isMouseOver = true;
        drawAll();
    }};

    canvas.onmouseout = function() {{
        isMouseOver = false;
        drawAll();
    }};

    // Her şeyi baştan çizme fonksiyonu (Canlı Önizleme için)
    function drawAll() {{
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        ctx.drawImage(img, 0, 0);
        
        // Sabitlenmiş (tıklanmış) kırmızı daireleri çiz
        for(var i=0; i<coords.length; i++) {{
            ctx.beginPath();
            ctx.arc(coords[i].x, coords[i].y, coords[i].r, 0, 2 * Math.PI, false);
            ctx.lineWidth = 4;
            ctx.strokeStyle = 'red';
            ctx.stroke();
            
            ctx.font = "bold 24px Arial";
            ctx.fillStyle = "red";
            ctx.fillText(wells[i], coords[i].x + coords[i].r + 5, coords[i].y);
        }}

        // Farenin ucundaki sarı önizleme dairesini çiz
        if (isMouseOver && coords.length < 16) {{
            var current_r = parseInt(radiusSlider.value);
            ctx.beginPath();
            ctx.arc(mouseX, mouseY, current_r, 0, 2 * Math.PI, false);
            ctx.lineWidth = 3;
            ctx.strokeStyle = 'yellow';
            ctx.setLineDash([5, 5]); // Kesik çizgili önizleme
            ctx.stroke();
            ctx.setLineDash([]); // Çizgi tipini sıfırla
        }}
    }}

    canvas.onclick = function(e) {{
        if (coords.length >= 16) return;
        var current_r = parseInt(radiusSlider.value);
        coords.push({{x: mouseX, y: mouseY, r: current_r}});
        
        document.getElementById('count').innerText = coords.length;
        if (coords.length < 16) {{
            document.getElementById('current-well').innerText = wells[coords.length];
        }} else {{
            document.getElementById('info-box').innerHTML = "<span style='color:green;'>Analiz yapılıyor...</span>";
            document.getElementById('controls').style.display = 'none'; // Sliderı gizle
            
            // [[x1,y1,r1], [x2,y2,r2]...] formatında listeye çevirip Python'a yolla
            var finalData = coords.map(c => [c.x, c.y, c.r]);
            resolvePromise(finalData); 
        }}
        drawAll();
    }};
</script>
'''

display(HTML(html_code))
clicked_coords_list = output.eval_js('promise') 

# JS'den gelen listeyi sözlüğe (dictionary) çevir
#  [X_Koordinatı, Y_Koordinatı, Yarıçap] formatında tutuluyor.
well_coords_and_radii = {kuyucuklar[i]: clicked_coords_list[i] for i in range(16)}
print("\nTüm koordinatlar ve daire boyutları başarıyla kaydedildi.")



def get_avg_rgb_dynamic(img, data):
    # data: [x, y, r]
    x, y, r = data
    mask = np.zeros(img.shape[:2], dtype="uint8")
    cv2.circle(mask, (x, y), r, 255, -1)
    mean_color = cv2.mean(img, mask=mask)[:3]
    return mean_color

standards = {
    0.031:  ['A1', 'A2'],
    0.062:  ['B1', 'B2'],
    0.125:  ['C1', 'C2'],
    0.250:  ['D1', 'D2'],
    0.500:  ['E1', 'E2'],
    0.750:  ['F1', 'F2'],
    1.0:    ['G1', 'G2']
}

concentrations = []
rgb_values = []

print("\n" + "="*40)
print(" RGB DEĞERLERİ")
print("="*40)

for conc, wells in standards.items():
    rgb_pair = []
    for well in wells:
        rgb = get_avg_rgb_dynamic(image, well_coords_and_radii[well])
        rgb_pair.append(rgb)
        x, y, r = well_coords_and_radii[well]
        print(f"[{well}] Konsantrasyon: {conc:<6} | R:{rgb[0]:.0f} G:{rgb[1]:.0f} B:{rgb[2]:.0f} (Boyut: {r}px)")
    
    avg_rgb = np.mean(rgb_pair, axis=0)
    concentrations.append(conc)
    rgb_values.append(avg_rgb)

concentrations = np.array(concentrations)
rgb_values = np.array(rgb_values)

# MODEL EĞİTİMİ
model = LinearRegression()
model.fit(rgb_values, concentrations)
y_pred = model.predict(rgb_values)
r2 = r2_score(concentrations, y_pred)

print("\n" + "="*40)
print("    REGRESYON SONUÇLARI")
print("="*40)
print(f"Model R² Skoru : {r2:.4f}")

# GRAFİK
plt.figure(figsize=(8, 6))
plt.scatter(concentrations, y_pred, color='darkblue', s=100, label='Standartlar (Eğitim Verisi)', zorder=3)
plt.plot([concentrations.min(), concentrations.max()], [concentrations.min(), concentrations.max()], 'r--', linewidth=2, label='İdeal Çizgi (Gerçek = Tahmin)')
plt.xlabel('Gerçek Konsantrasyon (BSA mg/mL)', fontweight='bold')
plt.ylabel('Modelin Tahmin Ettiği Konsantrasyon', fontweight='bold')
plt.title(f'RGB Lineer Regresyon Başarısı (R² = {r2:.3f})', fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# H1 ve H2 (BİLİNMEYENLERİN) TAHMİNİ
unknown_wells = ['H1', 'H2']
unknown_rgbs = []
for well in unknown_wells:
    rgb = get_avg_rgb_dynamic(image, well_coords_and_radii[well])
    unknown_rgbs.append(rgb)

unknown_preds = model.predict(np.array(unknown_rgbs))

print("\n" + "="*40)
print("  BİLİNMEYEN NUMUNE TAHMİNLERİ (H1, H2)")
print("="*40)
for i, well in enumerate(unknown_wells):
    pred = max(0, unknown_preds[i]) 
    print(f"[{well}] Tahmini Konsantrasyon: {pred:.4f} mg/mL")